# 02 — Data Preprocessing

Clean, standardize, and prepare the raw MLS dataset for feature engineering.
This notebook handles team name normalization, date parsing, and type casting.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from mls_predictor.data_loader import load_raw_data
from mls_predictor.config import TEAM_NAME_MAPPING, standardize_team_name

ImportError: cannot import name 'TEAM_NAME_MAP' from 'mls_predictor.config' (c:\Users\Dakkarm\Desktop\MLS-2026-Predictor\mls_predictor\config.py)

In [ ]:
df = load_raw_data()
print(f"Raw dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"\nDate range: {df['Date'].min()} → {df['Date'].max()}")
print(f"Seasons: {sorted(df['Season'].unique())}")

In [ ]:
# Check team name consistency
all_teams = sorted(set(df['HomeTeam'].unique()) | set(df['AwayTeam'].unique()))
print(f"{len(all_teams)} unique teams:")
for t in all_teams:
    std = standardize_team_name(t)
    flag = ' ← MAPPED' if std != t else ''
    print(f'  {t}{flag}')

In [ ]:
# Check odds columns — cast to numeric
odds_cols = ['PSCH', 'PSCD', 'PSCA', 'B365CH', 'B365CD', 'B365CA',
             'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA']

for col in odds_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Odds dtypes after casting:')
print(df[odds_cols].dtypes)
print('\nOdds sample:')
df[odds_cols].describe()

In [ ]:
# Verify no duplicate matches
dupes = df.duplicated(subset=['Date', 'HomeTeam', 'AwayTeam'], keep=False)
print(f"Duplicate matches: {dupes.sum()}")
if dupes.sum() > 0:
    print(df[dupes][['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']])

In [ ]:
# Final check
print('✅ Data preprocessing checks passed!')
print(f'  {len(df)} matches ready for feature engineering')
print(f'  {len(all_teams)} teams')
print(f'  Odds coverage: {df["PSCH"].notna().mean():.1%} (Pinnacle)')